# 00 · Tour pelo Databricks

*O mínimo de Databricks para entender (e explicar) a aula*

> **FIAP Pós-Tech · Fase 5 (MLOps)** — Recomendação de notícias com MLOps no Databricks.
> Matérias **reais** (sitemap do O Globo) · usuários e cliques **sintéticos**.

## O que é o Databricks, em uma frase?

Uma plataforma onde **dados, código, modelos e agendamentos vivem juntos e governados**:
você guarda tabelas, roda notebooks em Spark/Python, registra experimentos e modelos
e agenda pipelines — tudo no mesmo lugar, com o mesmo controle de acesso.

| Ideia | No Databricks | Analogia |
|---|---|---|
| Onde ficam os dados | **Unity Catalog**: `catálogo.schema.tabela` | banco → schema → tabela |
| Arquivos brutos | **Volume**: `/Volumes/catálogo/schema/volume/` | um "bucket" governado |
| Tabelas | **Delta Lake** (Parquet + histórico + ACID) | tabela de banco com `git log` |
| Rodar código | **Notebook** em **serverless compute** | Jupyter, mas sem gerenciar máquina |
| Experimentos | **MLflow Experiments** | planilha automática de treinos |
| Modelos | **Models** no Unity Catalog (versões + aliases) | "releases" do modelo |
| Agendar | **Jobs** (Jobs & Pipelines) | cron com DAG, retries e logs |
| Infra como código | **Asset Bundles** (`databricks.yml`) | Terraform do projeto |

## Onde clicar (menu lateral esquerdo)

| Menu | O que vamos ver nesta aula |
|---|---|
| **Workspace** | pasta `.bundle/news-recsys-databricks/...` com notebooks e `src/` (deploy do bundle) |
| **Catalog** | `workspace` → `news_recsys_dev` → tabelas `bronze_*`, `silver_*`, `gold_*` e o Volume `raw` |
| **Jobs & Pipelines** | job `[dev ...] news-recsys-pipeline` (6 tasks) — histórico de execuções |
| **Experiments** | `news_recsys_...` — cada treino é uma *run* com parâmetros e métricas |
| **Models** (dentro do Catalog) | `news_recommender` — versões e alias `@champion` |
| **SQL Warehouses** | o warehouse que o app Streamlit usa para ler a tabela gold |

> **Dica para a aula:** deixe essas abas abertas antes de começar.

## O que vamos construir

```
GitHub Actions (cron) ──► sitemap O Globo ──► JSON ──► Volume raw
                                                         │
   01 bronze_articles ◄──────────────────────────────────┘
         │
   02 silver_users + silver_interactions   (cliques SINTÉTICOS)
         │
   03 treino: popularidade × item-CF × SVD   ──► MLflow (runs) ──► Unity Catalog (versão N)
         │
   04 quality gate: N supera o @champion?  ──► sim: alias @champion = N
         │
   05 inferência batch com @champion ──► gold_user_recommendations
         │                                        │
   06 monitoramento (cobertura, drift)           └──► App Streamlit (Community Cloud)
```

Os notebooks 01→06 são as **tasks do Job**. O GitHub Actions faz CI (testes), CD
(deploy do bundle) e o gatilho diário. O Terraform cria o repositório e os secrets.

In [ ]:
# Célula 1 — dependências (sempre a primeira: o restartPython apaga as variáveis)
%pip install -q -r ../requirements-databricks.txt
dbutils.library.restartPython()

In [ ]:
# Célula 2 — src/ no sys.path + widgets (= parâmetros do Job) -> settings
import sys
from pathlib import Path


def _find_src() -> Path:
    candidates = [Path.cwd().parent / "src"]
    try:  # fallback: caminho do notebook no workspace
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        candidates.append(Path("/Workspace" + ctx.notebookPath().get()).parent.parent / "src")
    except Exception:  # noqa: BLE001
        pass
    for c in candidates:
        if (c / "news_recsys").is_dir():
            return c.resolve()
    raise FileNotFoundError(f"src/ não encontrado em {candidates}")


SRC = _find_src()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
ROOT = SRC.parent

from news_recsys.config import Settings

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "news_recsys_dev")
dbutils.widgets.text("n_users", "300")
dbutils.widgets.text("seed", "42")

settings = Settings(_env_file=None).with_overrides(
    catalog=dbutils.widgets.get("catalog"),
    schema_name=dbutils.widgets.get("schema"),
    n_users=dbutils.widgets.get("n_users"),
    seed=dbutils.widgets.get("seed"),
)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {settings.catalog}.{settings.schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {settings.catalog}.{settings.schema_name}.{settings.volume}")
print(f"schema ..... {settings.catalog}.{settings.schema_name}")
print(f"modelo ..... {settings.model_fqn}")
print(f"usuários ... {settings.n_users} (seed={settings.seed})")

### Execução: quem sou eu e o que existe no catálogo?

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
print("usuário:  ", w.current_user.me().user_name)
print("workspace:", w.config.host)
spark.sql(f"SHOW TABLES IN {settings.catalog}.{settings.schema_name}").toPandas()

### Execução: Spark × pandas em 10 segundos

`spark` é um DataFrame **distribuído** (e preguiçoso: só executa no `.toPandas()`,
`.count()`, `display()`...). Nossos dados são pequenos, então treinamos com pandas e
scikit-learn no *driver* — e usamos Spark para ler/gravar tabelas Delta.

In [ ]:
df = spark.range(1_000_000).selectExpr("id", "id % 7 AS resto")
print(type(df).__name__, "->", df.count(), "linhas (calculado no cluster serverless)")
df.groupBy("resto").count().orderBy("resto").toPandas()

### O que percebemos?

- Tudo tem **endereço no catálogo** (`workspace.news_recsys_dev.tabela`) e permissões.
- Não criamos nenhuma máquina: serverless liga e desliga sozinho.
- O schema e o Volume foram criados pelo bootstrap (`CREATE ... IF NOT EXISTS`) —
  rodar de novo não quebra nada (**idempotência**).

**Próximo:** `01_ingestao_bronze`.